# Tanseek — Connected data & conflict-engine starter

This notebook uses the **existing synthetic CSVs** that follow the PostgreSQL schema. It builds relational views and a deterministic constraint checker as a starting point for Menna's scheduling model. Run top to bottom. Keep the `Tanseek_CSV_Data` folder and `Tanseek_constraint_cases.csv` beside this notebook. No database or internet is required.

Policy: Saturday–Wednesday, 09:00–17:00, four contiguous 120-minute slots. Availability is due August 31 each year. Data is synthetic; dates in this fixture are for the 2027 term.

In [1]:
from pathlib import Path
from collections import defaultdict
import pandas as pd

ROOT = Path.cwd()
DATA = ROOT / "Tanseek_CSV_Data"
CASES = ROOT / "Tanseek_constraint_cases.csv"
assert DATA.is_dir() and CASES.is_file(), "Extract the ZIP beside this notebook first."
tables = {p.stem: pd.read_csv(p, encoding="utf-8-sig") for p in DATA.glob("*.csv")}
cases = pd.read_csv(CASES, encoding="utf-8-sig").fillna("")
pd.DataFrame([(name, len(frame), ", ".join(frame.columns)) for name, frame in sorted(tables.items())], columns=["table", "rows", "columns"])


,table,rows,columns
0,academic_terms,1,"id, name, starts_on, ends_on, state, availabil..."
1,accounts,18,"id, email, full_name, role, state, home_depart..."
2,allocations,1,"id, term_id, version_id, section_id, requireme..."
3,availability_slots,280,"submission_id, term_id, slot_id, kind"
4,availability_submissions,15,"id, term_id, instructor_id, state, confirmed_a..."
5,courses,5,"id, department_id, code, title, created_by"
6,departments,2,"id, code, name"
7,equipment,3,"id, name"
8,required_equipment,5,"requirement_id, equipment_id, quantity"
9,room_equipment,20,"room_id, equipment_id, quantity"


## Relationship checks

These verify the foreign keys needed by the model. A missing reference raises an error before a schedule is evaluated.

In [2]:
relations = [
    ("courses", "department_id", "departments", "id"),
    ("sections", "course_id", "courses", "id"),
    ("sections", "term_id", "academic_terms", "id"),
    ("section_groups", "section_id", "sections", "id"),
    ("section_groups", "group_id", "student_groups", "id"),
    ("section_instructors", "section_id", "sections", "id"),
    ("section_instructors", "instructor_id", "accounts", "id"),
    ("section_instructors", "requirement_id", "session_requirements", "id"),
    ("required_equipment", "requirement_id", "session_requirements", "id"),
    ("room_equipment", "room_id", "rooms", "id"),
    ("availability_slots", "submission_id", "availability_submissions", "id"),
    ("availability_slots", "slot_id", "time_slots", "id"),
    ("allocations", "section_id", "sections", "id"),
    ("allocations", "start_slot_id", "time_slots", "id"),
]
integrity = []
for child, column, parent, key in relations:
    missing = set(tables[child][column].dropna()) - set(tables[parent][key])
    integrity.append((child + "." + column, parent + "." + key, len(missing)))
integrity = pd.DataFrame(integrity, columns=["foreign_key", "references", "missing_ids"])
assert (integrity.missing_ids == 0).all(), integrity[integrity.missing_ids > 0]
integrity


,foreign_key,references,missing_ids
0,courses.department_id,departments.id,0
1,sections.course_id,courses.id,0
2,sections.term_id,academic_terms.id,0
3,section_groups.section_id,sections.id,0
4,section_groups.group_id,student_groups.id,0
5,section_instructors.section_id,sections.id,0
6,section_instructors.instructor_id,accounts.id,0
7,section_instructors.requirement_id,session_requirements.id,0
8,required_equipment.requirement_id,session_requirements.id,0
9,room_equipment.room_id,rooms.id,0


## Connected views

Aggregate many-to-many links *before* joining them to sections, so group counts and equipment quantities do not get multiplied. `section_view` gives one row per section. `candidate_view` gives one row per eligible section/instructor/requirement/room/slot combination after the hard checks below.

In [3]:
group_links = tables["section_groups"].merge(
    tables["student_groups"][["id", "student_count", "name"]], left_on="group_id", right_on="id", validate="many_to_one"
)
group_summary = group_links.groupby("section_id", as_index=False).agg(
    group_ids=("group_id", list), student_count=("student_count", "sum"), group_names=("name", list)
)
section_view = (tables["sections"].merge(tables["courses"][["id", "code", "title", "department_id"]],
             left_on="course_id", right_on="id", validate="many_to_one", suffixes=("", "_course"))
             .merge(group_summary, left_on="id", right_on="section_id", validate="one_to_one", suffixes=("", "_groups")))
assert len(section_view) == len(tables["sections"])
section_view[["id", "code", "group_names", "student_count", "department_id"]].head()


,id,code,group_names,student_count,department_id
0,1,AI301-S1,"[AI301-S1-G1, AI301-S1-G2]",38,1
1,2,AI301-S2,"[AI301-S2-G1, AI301-S2-G2]",40,1
2,3,AI301-S3,"[AI301-S3-G1, AI301-S3-G2]",36,1
3,4,AI301-S4,"[AI301-S4-G1, AI301-S4-G2]",38,1
4,5,AI301-S5,"[AI301-S5-G1, AI301-S5-G2]",40,1


In [4]:
# Compact maps keep the checker readable, while the DataFrames above remain available for analysis.
def by_id(name): return tables[name].set_index("id").to_dict("index")
sections, courses, requirements = [by_id(n) for n in ("sections", "courses", "session_requirements")]
rooms, slots, accounts = [by_id(n) for n in ("rooms", "time_slots", "accounts")]
groups_by_section = {int(r.section_id): set(map(int, r.group_ids)) for r in group_summary.itertuples()}
students_by_section = {int(r.section_id): int(r.student_count) for r in group_summary.itertuples()}
required = {(int(r.requirement_id), int(r.equipment_id)): int(r.quantity) for r in tables["required_equipment"].itertuples()}
available = {(int(r.room_id), int(r.equipment_id)): int(r.quantity) for r in tables["room_equipment"].itertuples()}
eligible = {(int(r.section_id), int(r.requirement_id), int(r.instructor_id)) for r in tables["section_instructors"].itertuples()}
submissions = tables["availability_submissions"].sort_values(["revision", "id"]).drop_duplicates(["term_id", "instructor_id"], keep="last")
submission_by_staff = {(int(r.term_id), int(r.instructor_id)): r for r in submissions.itertuples(index=False)}
availability = {(int(r.submission_id), int(r.slot_id)): r.kind for r in tables["availability_slots"].itertuples(index=False)}
allocations = tables["allocations"].merge(tables["time_slots"][["id", "weekday", "starts_at"]],
    left_on="start_slot_id", right_on="id", validate="many_to_one", suffixes=("", "_slot"))

def minutes(value):
    hh, mm = str(value).split(":")[:2]
    return int(hh) * 60 + int(mm)

def overlaps(start1, end1, start2, end2):
    return max(start1, start2) < min(end1, end2)


## Hard constraints

`check_allocation()` returns every reason in a stable order. The primary reason follows the CSV cases' intended priority. `fixture_group_ids` is a case-local override and never changes the shared source data. An unlisted availability slot is treated as unknown and blocked.

In [5]:
# -------------------------
# Hard Constraint Priority
# -------------------------

PRIORITY = [
    "NO_WORKING_SLOT",
    "TERM_HOLIDAY",
    "ROOM_CLOSED",
    "INVALID_DURATION",
    "AVAILABILITY_NOT_CONFIRMED",
    "STAFF_UNAVAILABLE",
    "ROOM_CONFLICT",
    "STAFF_CONFLICT",
    "GROUP_CONFLICT",
    "ROOM_TYPE_MISMATCH",
    "CAPACITY_SHORTAGE",
    "EQUIPMENT_SHORTAGE",
    "INELIGIBLE_INSTRUCTOR",
    "INVALID_REFERENCE"
]


# -------------------------
# Allocation Constraint Checker
# -------------------------

def check_allocation(
    term_id,
    section_id,
    requirement_id,
    instructor_id,
    room_id,
    weekday,
    starts_at,
    ends_at,
    fixture_group_ids=None,
    existing=None,
    session_date=None
):
    # Use current allocations unless another
    # allocation set is explicitly supplied
    if existing is None:
        existing = allocations

    # Normalize IDs
    term_id = int(term_id)
    section_id = int(section_id)
    requirement_id = int(requirement_id)
    instructor_id = int(instructor_id)
    room_id = int(room_id)
    weekday = int(weekday)

    reasons = set()

    # -------------------------
    # Reference validation
    # -------------------------
    if any(
        key not in mapping
        for key, mapping in [
            (section_id, sections),
            (requirement_id, requirements),
            (room_id, rooms),
            (instructor_id, accounts)
        ]
    ):
        return {
            "primary_result": "INVALID_REFERENCE",
            "reasons": ["INVALID_REFERENCE"],
            "slot_id": None
        }

    section = sections[section_id]
    req = requirements[requirement_id]
    room = rooms[room_id]

    if (
        int(section["term_id"]) != term_id
        or int(req["term_id"]) != term_id
        or int(section["course_id"])
        != int(req["course_id"])
    ):
        reasons.add("INVALID_REFERENCE")

    # -------------------------
    # Instructor eligibility
    # -------------------------
    if (
        section_id,
        requirement_id,
        instructor_id
    ) not in eligible:
        reasons.add("INELIGIBLE_INSTRUCTOR")

    # -------------------------
    # Time validation
    # -------------------------
    start = minutes(starts_at)
    end = minutes(ends_at)

    # -------------------------
    # Date-specific constraints
    # Only applied when a concrete
    # session date is provided.
    # -------------------------
    if session_date is not None:

        if is_term_holiday(
            term_id,
            session_date
        ):
            reasons.add(
                "TERM_HOLIDAY"
            )

        if is_room_closed(
            room_id,
            session_date,
            starts_at,
            ends_at
        ):
            reasons.add(
                "ROOM_CLOSED"
            )

    matching = [
        (slot_id, row)
        for slot_id, row in slots.items()
        if int(row["term_id"]) == term_id
        and int(row["weekday"]) == weekday
        and minutes(row["starts_at"]) == start
    ]

    slot_id = (
        matching[0][0]
        if matching
        else None
    )

    if (
        not matching
        or start < 9 * 60
    ):
        reasons.add(
            "NO_WORKING_SLOT"
        )

    if (
        end - start
        != int(req["duration_minutes"])
        or (
            matching
            and end
            != minutes(
                matching[0][1]["ends_at"]
            )
        )
    ):
        reasons.add(
            "INVALID_DURATION"
        )

    # -------------------------
    # Instructor availability
    # -------------------------
    submission = (
        submission_by_staff.get(
            (term_id, instructor_id)
        )
    )

    if (
        submission is None
        or submission.state
        != "CONFIRMED"
    ):
        reasons.add(
            "AVAILABILITY_NOT_CONFIRMED"
        )

    elif (
        slot_id is not None
        and availability.get(
            (
                int(submission.id),
                slot_id
            )
        )
        not in (
            "AVAILABLE",
            "PREFERRED"
        )
    ):
        reasons.add(
            "STAFF_UNAVAILABLE"
        )

    # -------------------------
    # Room validation
    # -------------------------
    if (
        room["kind"]
        != req["required_room_kind"]
        or not bool(room["active"])
    ):
        reasons.add(
            "ROOM_TYPE_MISMATCH"
        )

    if (
        students_by_section.get(
            section_id,
            0
        )
        > int(room["capacity"])
    ):
        reasons.add(
            "CAPACITY_SHORTAGE"
        )

    # -------------------------
    # Equipment validation
    # -------------------------
    for (
        req_id,
        equipment_id
    ), quantity in required.items():

        if (
            req_id == requirement_id
            and available.get(
                (
                    room_id,
                    equipment_id
                ),
                0
            )
            < quantity
        ):
            reasons.add(
                "EQUIPMENT_SHORTAGE"
            )

    # -------------------------
    # Student groups
    # -------------------------
    candidate_groups = (
        set(fixture_group_ids)
        if fixture_group_ids is not None
        else groups_by_section.get(
            section_id,
            set()
        )
    )

    # -------------------------
    # Existing allocation conflicts
    # -------------------------
    for old in existing.itertuples(
        index=False
    ):

        if int(old.term_id) != term_id:
            continue

        if int(old.weekday) != weekday:
            continue

        # Same-section conflicts are
        # handled in check_candidate_strict()
        if (
            int(old.section_id)
            == section_id
        ):
            continue

        if not overlaps(
            start,
            end,
            minutes(old.starts_at),
            minutes(old.ends_at)
        ):
            continue

        # Room conflict
        if (
            int(old.room_id)
            == room_id
        ):
            reasons.add(
                "ROOM_CONFLICT"
            )

        # Instructor conflict
        if (
            int(old.instructor_id)
            == instructor_id
        ):
            reasons.add(
                "STAFF_CONFLICT"
            )

        # Student-group conflict
        old_groups = (
            groups_by_section.get(
                int(old.section_id),
                set()
            )
        )

        if (
            candidate_groups
            & old_groups
        ):
            reasons.add(
                "GROUP_CONFLICT"
            )

    # -------------------------
    # Final ordered result
    # -------------------------
    ordered = [
        reason
        for reason in PRIORITY
        if reason in reasons
    ]

    return {
        "primary_result": (
            ordered[0]
            if ordered
            else "FEASIBLE"
        ),

        "reasons": ordered,

        "slot_id": slot_id
    }


print(
    "Allocation constraint checker ready "
)

Allocation constraint checker ready 


## Work through the supplied cases

The CSV's `student_count`, `room_capacity`, and equipment quantities are explanatory snapshots. The checker reads those facts from the related CSV tables. The group collision fixture temporarily shares a group with section 1.

In [6]:
results = []
for c in cases.itertuples(index=False):
    fixture = None
    if c.case_id == "GROUP_COLLISION":
        fixture = (groups_by_section[int(c.section_id)] - {max(groups_by_section[int(c.section_id)])}) | {min(groups_by_section[1])}
    result = check_allocation(c.term_id, c.section_id, c.requirement_id, c.instructor_id,
                              c.room_id, c.weekday, c.starts_at, c.ends_at, fixture_group_ids=fixture)
    results.append({"case_id": c.case_id, "expected": c.expected_primary_result,
                    "actual": result["primary_result"], "all_reasons": result["reasons"],
                    "pass": c.expected_primary_result == result["primary_result"]})
case_results = pd.DataFrame(results)
display(case_results)
assert case_results["pass"].all(), case_results.loc[~case_results["pass"]]


,case_id,expected,actual,all_reasons,pass
0,VALID_BASE,FEASIBLE,FEASIBLE,[],True
1,ROOM_COLLISION,ROOM_CONFLICT,ROOM_CONFLICT,[ROOM_CONFLICT],True
2,STAFF_COLLISION,STAFF_CONFLICT,STAFF_CONFLICT,[STAFF_CONFLICT],True
3,GROUP_COLLISION,GROUP_CONFLICT,GROUP_CONFLICT,[GROUP_CONFLICT],True
4,UNCONFIRMED,AVAILABILITY_NOT_CONFIRMED,AVAILABILITY_NOT_CONFIRMED,[AVAILABILITY_NOT_CONFIRMED],True
5,STAFF_UNAVAILABLE,STAFF_UNAVAILABLE,STAFF_UNAVAILABLE,[STAFF_UNAVAILABLE],True
6,EQUIPMENT_SHORTAGE,EQUIPMENT_SHORTAGE,EQUIPMENT_SHORTAGE,[EQUIPMENT_SHORTAGE],True
7,CAPACITY_SHORTAGE,CAPACITY_SHORTAGE,CAPACITY_SHORTAGE,[CAPACITY_SHORTAGE],True
8,WRONG_ROOM_TYPE,ROOM_TYPE_MISMATCH,ROOM_TYPE_MISMATCH,"[ROOM_TYPE_MISMATCH, EQUIPMENT_SHORTAGE]",True
9,BEYOND_HOURS,INVALID_DURATION,INVALID_DURATION,[INVALID_DURATION],True


## Generate feasible candidates and rank alternatives

The demo generates candidate combinations for one section and one weekly session. It excludes hard conflicts and prefers a `PREFERRED` availability slot, then an earlier time. For a full timetable solver, add weekly session instances, teacher workload, room closure/holiday tables, and an optimizer that chooses a compatible set *jointly*. This starter does not claim to solve the global optimization problem.

In [7]:
def rank_candidates(section_id, requirement_id, limit=10):
    sec = sections[int(section_id)]
    term_id = int(sec["term_id"])
    staff = sorted({i for s, r, i in eligible if s == int(section_id) and r == int(requirement_id)})
    rows = []
    for instructor_id in staff:
        submission = submission_by_staff.get((term_id, instructor_id))
        for slot_id, slot in slots.items():
            if int(slot["term_id"]) != term_id: continue
            for room_id in rooms:
                result = check_allocation(term_id, section_id, requirement_id, instructor_id,
                    room_id, slot["weekday"], slot["starts_at"], slot["ends_at"])
                if result["primary_result"] != "FEASIBLE": continue
                kind = availability.get((int(submission.id), slot_id), "UNKNOWN") if submission else "UNKNOWN"
                rows.append({"section_id": section_id, "requirement_id": requirement_id,
                             "instructor_id": instructor_id, "room_id": room_id,
                             "slot_id": slot_id, "weekday": slot["weekday"],
                             "starts_at": slot["starts_at"], "availability": kind,
                             "preference_rank": 0 if kind == "PREFERRED" else 1})
    if not rows: return pd.DataFrame(columns=["section_id", "requirement_id", "instructor_id", "room_id", "slot_id", "weekday", "starts_at", "availability", "preference_rank"])
    return pd.DataFrame(rows).sort_values(["preference_rank", "weekday", "starts_at", "room_id"]).head(limit).reset_index(drop=True)

candidate_view = rank_candidates(section_id=2, requirement_id=1)
candidate_view


,section_id,requirement_id,instructor_id,room_id,slot_id,weekday,starts_at,availability,preference_rank
0,2,1,5,1,9,1,09:00:00,AVAILABLE,1
1,2,1,5,3,9,1,09:00:00,AVAILABLE,1
2,2,1,5,4,9,1,09:00:00,AVAILABLE,1
3,2,1,5,5,9,1,09:00:00,AVAILABLE,1
4,2,1,5,6,9,1,09:00:00,AVAILABLE,1
5,2,1,5,7,9,1,09:00:00,AVAILABLE,1
6,2,1,5,8,9,1,09:00:00,AVAILABLE,1
7,2,1,5,9,9,1,09:00:00,AVAILABLE,1
8,2,1,4,1,10,1,11:00:00,AVAILABLE,1
9,2,1,5,1,10,1,11:00:00,AVAILABLE,1


In [8]:
from ortools.sat.python import cp_model
from collections import defaultdict, Counter
import json

print("OR-Tools ready ")

OR-Tools ready 


In [9]:
def run_fixture_tests():
    results = []

    for c in cases.itertuples(index=False):
        fixture = None

        if c.case_id == "GROUP_COLLISION":
            fixture = (
                groups_by_section[int(c.section_id)]
                - {max(groups_by_section[int(c.section_id)])}
            ) | {min(groups_by_section[1])}

        result = check_allocation(
            c.term_id,
            c.section_id,
            c.requirement_id,
            c.instructor_id,
            c.room_id,
            c.weekday,
            c.starts_at,
            c.ends_at,
            fixture_group_ids=fixture
        )

        results.append({
            "case_id": c.case_id,
            "expected": c.expected_primary_result,
            "actual": result["primary_result"],
            "all_reasons": result["reasons"],
            "pass": c.expected_primary_result == result["primary_result"]
        })

    result_df = pd.DataFrame(results)

    display(result_df)

    assert result_df["pass"].all(), \
        result_df.loc[~result_df["pass"]]

    print("All 11 fixture tests passed ✅")

    return result_df

In [10]:
# -------------------------
# Schedule Version Context
# -------------------------

TERM_ID = int(
    tables["sections"]["term_id"]
    .mode()
    .iloc[0]
)

schedule_versions_df = (
    tables["schedule_versions"]
    .copy()
)

term_versions = (
    schedule_versions_df[
        schedule_versions_df["term_id"].astype(int)
        == TERM_ID
    ]
    .sort_values(
        ["version_number", "id"]
    )
    .reset_index(drop=True)
)

display(
    term_versions[
        [
            "id",
            "term_id",
            "version_number",
            "name",
            "state"
        ]
    ]
)

,id,term_id,version_number,name,state
0,1,1,1,Draft 1,DRAFT


In [11]:
def get_latest_version_by_state(
    term_id,
    state
):
    matches = schedule_versions_df[
        (
            schedule_versions_df["term_id"]
            .astype(int)
            == int(term_id)
        )
        &
        (
            schedule_versions_df["state"]
            == state
        )
    ]

    if matches.empty:
        return None

    return (
        matches
        .sort_values(
            ["version_number", "id"]
        )
        .iloc[-1]
    )


published_version = get_latest_version_by_state(
    TERM_ID,
    "PUBLISHED"
)

working_version = get_latest_version_by_state(
    TERM_ID,
    "DRAFT"
)


print(
    "Published version:",
    None
    if published_version is None
    else published_version["name"]
)

print(
    "Working version:",
    None
    if working_version is None
    else working_version["name"]
)

Published version: None
Working version: Draft 1


In [12]:
def get_version_allocations(
    version_id
):
    if version_id is None:
        return allocations.iloc[0:0].copy()

    return (
        allocations[
            allocations["version_id"].astype(int)
            == int(version_id)
        ]
        .copy()
        .reset_index(drop=True)
    )


published_allocations = (
    get_version_allocations(
        None
        if published_version is None
        else int(published_version["id"])
    )
)

working_allocations = (
    get_version_allocations(
        None
        if working_version is None
        else int(working_version["id"])
    )
)


print(
    "Published allocations:",
    len(published_allocations)
)

print(
    "Working allocations:",
    len(working_allocations)
)

Published allocations: 0
Working allocations: 1


In [13]:
if working_version is not None:
    solver_existing_allocations = (
        working_allocations.copy()
    )

elif published_version is not None:
    solver_existing_allocations = (
        published_allocations.copy()
    )

else:
    solver_existing_allocations = (
        allocations.iloc[0:0].copy()
    )


print(
    "Solver existing allocations:",
    len(solver_existing_allocations)
)

Solver existing allocations: 1


In [14]:
published_snapshot = (
    published_allocations
    .copy(deep=True)
)


def verify_published_immutable():
    if published_version is None:
        print(
            "No published version yet — "
            "immutability check skipped "
        )
        return

    current_published = (
        get_version_allocations(
            int(published_version["id"])
        )
    )

    pd.testing.assert_frame_equal(
        published_snapshot.reset_index(
            drop=True
        ),
        current_published.reset_index(
            drop=True
        )
    )

    print(
        "Published schedule remained immutable "
    )


verify_published_immutable()

No published version yet — immutability check skipped 


In [15]:
# -------------------------
# Optional Holidays & Room Closures
# -------------------------

term_holidays = tables.get(
    "term_holidays",
    pd.DataFrame(
        columns=[
            "id",
            "term_id",
            "holiday_date",
            "reason"
        ]
    )
).copy()

room_closures = tables.get(
    "room_closures",
    pd.DataFrame(
        columns=[
            "id",
            "room_id",
            "starts_at",
            "ends_at",
            "reason"
        ]
    )
).copy()


if not term_holidays.empty:
    term_holidays["holiday_date"] = (
        pd.to_datetime(
            term_holidays["holiday_date"]
        ).dt.date
    )


if not room_closures.empty:
    room_closures["starts_at_ts"] = (
        pd.to_datetime(
            room_closures["starts_at"],
            utc=True
        ).dt.tz_convert("Africa/Cairo")
    )

    room_closures["ends_at_ts"] = (
        pd.to_datetime(
            room_closures["ends_at"],
            utc=True
        ).dt.tz_convert("Africa/Cairo")
    )


print(
    "Term holidays loaded:",
    len(term_holidays)
)

print(
    "Room closures loaded:",
    len(room_closures)
)

Term holidays loaded: 0
Room closures loaded: 0


In [16]:
# -------------------------
# Holiday / Room Closure Helpers
# -------------------------

def is_term_holiday(
    term_id,
    session_date,
    holidays_df=None
):
    df = (
        term_holidays
        if holidays_df is None
        else holidays_df
    )

    if session_date is None:
        return False

    if df.empty:
        return False

    session_date = pd.Timestamp(
        session_date
    ).date()

    match = df[
        (df["term_id"].astype(int) == int(term_id))
        &
        (df["holiday_date"] == session_date)
    ]

    return not match.empty


def is_room_closed(
    room_id,
    session_date,
    starts_at,
    ends_at,
    closures_df=None
):
    df = (
        room_closures
        if closures_df is None
        else closures_df
    )

    if session_date is None:
        return False

    if df.empty:
        return False

    candidate_start = pd.Timestamp(
        f"{session_date} {starts_at}",
        tz="Africa/Cairo"
    )

    candidate_end = pd.Timestamp(
        f"{session_date} {ends_at}",
        tz="Africa/Cairo"
    )

    room_rows = df[
        df["room_id"].astype(int)
        == int(room_id)
    ]

    for closure in room_rows.itertuples(
        index=False
    ):
        if (
            candidate_start < closure.ends_at_ts
            and candidate_end > closure.starts_at_ts
        ):
            return True

    return False


print("Holiday / closure helpers ready ")

Holiday / closure helpers ready 


In [17]:
# -------------------------
# Published Change Request Validator
# -------------------------

def evaluate_published_change_request(
    allocation_id,
    session_date,
    proposed_room_id=None,
    proposed_instructor_id=None,
    proposed_start_slot_id=None
):
    # A real published version is required
    if published_version is None:
        return {
            "status": "NO_PUBLISHED_VERSION",
            "primary_result": None,
            "reasons": [
                "NO_PUBLISHED_VERSION"
            ]
        }

    # Read only from the immutable published snapshot
    target_rows = published_snapshot[
        published_snapshot["id"].astype(int)
        == int(allocation_id)
    ]

    if target_rows.empty:
        return {
            "status": "ALLOCATION_NOT_FOUND",
            "primary_result": "INVALID_REFERENCE",
            "reasons": [
                "INVALID_REFERENCE"
            ]
        }

    target = target_rows.iloc[0]

    term_id = int(target["term_id"])
    section_id = int(target["section_id"])
    requirement_id = int(
        target["requirement_id"]
    )

    instructor_id = (
        int(target["instructor_id"])
        if proposed_instructor_id is None
        else int(proposed_instructor_id)
    )

    room_id = (
        int(target["room_id"])
        if proposed_room_id is None
        else int(proposed_room_id)
    )

    start_slot_id = (
        int(target["start_slot_id"])
        if proposed_start_slot_id is None
        else int(proposed_start_slot_id)
    )

    if start_slot_id not in slots:
        return {
            "status": "BLOCKED",
            "primary_result": "INVALID_REFERENCE",
            "reasons": [
                "INVALID_REFERENCE"
            ]
        }

    slot = slots[start_slot_id]

    session_date_ts = pd.Timestamp(
        session_date
    )

    weekday = int(
        session_date_ts.isoweekday()
    )

    # Proposed slot must belong to
    # the same real weekday as session_date
    if int(slot["weekday"]) != weekday:
        return {
            "status": "BLOCKED",
            "primary_result": "NO_WORKING_SLOT",
            "reasons": [
                "NO_WORKING_SLOT"
            ]
        }

    # Everything in the published version remains fixed
    # except the allocation being changed.
    immutable_existing = published_snapshot[
        published_snapshot["id"].astype(int)
        != int(allocation_id)
    ].copy()

    result = check_candidate_strict(
        term_id=term_id,
        section_id=section_id,
        requirement_id=requirement_id,
        instructor_id=instructor_id,
        room_id=room_id,
        weekday=weekday,
        starts_at=slot["starts_at"],
        ends_at=slot["ends_at"],
        existing=immutable_existing,
        session_date=session_date
    )

    return {
        "status": (
            "FEASIBLE"
            if result["primary_result"]
            == "FEASIBLE"
            else "BLOCKED"
        ),

        "primary_result":
            result["primary_result"],

        "reasons":
            result["reasons"],

        "source_published_version_id":
            int(published_version["id"]),

        "source_allocation_id":
            int(allocation_id),

        "proposal": {
            "session_date":
                str(session_date),

            "weekday":
                weekday,

            "start_slot_id":
                start_slot_id,

            "room_id":
                room_id,

            "instructor_id":
                instructor_id
        }
    }


print(
    "Published change-request validator ready "
)

Published change-request validator ready 


In [18]:
# -------------------------
# Task 4 Date Constraint Tests
# -------------------------

TEST_DATE = "2027-10-03"

test_holidays = pd.DataFrame([
    {
        "id": 1,
        "term_id": TERM_ID,
        "holiday_date":
            pd.Timestamp(TEST_DATE).date(),
        "reason": "Synthetic holiday test"
    }
])


test_closures = pd.DataFrame([
    {
        "id": 1,
        "room_id": 1,
        "starts_at_ts":
            pd.Timestamp(
                f"{TEST_DATE} 09:30",
                tz="Africa/Cairo"
            ),
        "ends_at_ts":
            pd.Timestamp(
                f"{TEST_DATE} 10:30",
                tz="Africa/Cairo"
            ),
        "reason":
            "Synthetic closure test"
    }
])


assert is_term_holiday(
    TERM_ID,
    TEST_DATE,
    holidays_df=test_holidays
)


assert is_room_closed(
    room_id=1,
    session_date=TEST_DATE,
    starts_at="09:00",
    ends_at="11:00",
    closures_df=test_closures
)


assert not is_room_closed(
    room_id=1,
    session_date=TEST_DATE,
    starts_at="11:00",
    ends_at="13:00",
    closures_df=test_closures
)


print(
    "Task 4 holiday/closure tests passed "
)

Task 4 holiday/closure tests passed 


In [19]:
# -------------------------
# Published Change Request / Immutability Test
# -------------------------

def test_published_change_request():

    # Current synthetic data may not contain
    # a real PUBLISHED schedule yet.
    if published_version is None:

        result = evaluate_published_change_request(
            allocation_id=-1,
            session_date=TEST_DATE
        )

        assert (
            result["status"]
            == "NO_PUBLISHED_VERSION"
        )

        assert (
            result["reasons"]
            == ["NO_PUBLISHED_VERSION"]
        )

        print(
            "Published change-request gate test passed ✅"
        )

        return


    assert not published_snapshot.empty

    before = (
        published_snapshot
        .copy(deep=True)
        .reset_index(drop=True)
    )

    target = published_snapshot.iloc[0]

    # Pick a concrete date with the same weekday
    # as the published allocation.
    probe_date = pd.Timestamp(TEST_DATE)

    required_weekday = int(
        target["weekday"]
    )

    delta_days = (
        required_weekday
        - probe_date.isoweekday()
    ) % 7

    probe_date = (
        probe_date
        + pd.Timedelta(days=delta_days)
    ).date()


    result = evaluate_published_change_request(
        allocation_id=int(target["id"]),
        session_date=str(probe_date)
    )

    # A change request may be feasible or blocked,
    # but it must never mutate the published source.
    assert result["status"] in (
        "FEASIBLE",
        "BLOCKED"
    )

    after = (
        published_snapshot
        .copy(deep=True)
        .reset_index(drop=True)
    )

    pd.testing.assert_frame_equal(
        before,
        after
    )

    verify_published_immutable()

    print(
        "Published change-request immutability test passed ✅"
    )


test_published_change_request()

Published change-request gate test passed ✅


In [20]:
# -------------------------
# Section Requirements
# -------------------------

section_requirements_df = (
    tables["sections"][
        [
            "id",
            "term_id",
            "course_id",
            "code"
        ]
    ]
    .rename(
        columns={
            "id": "section_id",
            "code": "section_code"
        }
    )
    .merge(
        tables["session_requirements"][
            [
                "id",
                "term_id",
                "course_id",
                "kind",
                "sessions_per_week",
                "duration_minutes",
                "required_room_kind"
            ]
        ].rename(
            columns={
                "id": "requirement_id"
            }
        ),
        on=[
            "term_id",
            "course_id"
        ],
        how="inner"
    )
)


print(
    "Section requirements:",
    len(section_requirements_df)
)

display(
    section_requirements_df.head(10)
)

Section requirements: 25


,section_id,term_id,course_id,section_code,requirement_id,kind,sessions_per_week,duration_minutes,required_room_kind
0,1,1,1,AI301-S1,1,PRACTICAL,1,120,LAB
1,2,1,1,AI301-S2,1,PRACTICAL,1,120,LAB
2,3,1,1,AI301-S3,1,PRACTICAL,1,120,LAB
3,4,1,1,AI301-S4,1,PRACTICAL,1,120,LAB
4,5,1,1,AI301-S5,1,PRACTICAL,1,120,LAB
5,6,1,2,AI302-S1,2,PRACTICAL,1,120,LAB
6,7,1,2,AI302-S2,2,PRACTICAL,1,120,LAB
7,8,1,2,AI302-S3,2,PRACTICAL,1,120,LAB
8,9,1,2,AI302-S4,2,PRACTICAL,1,120,LAB
9,10,1,2,AI302-S5,2,PRACTICAL,1,120,LAB


In [21]:
weekly_session_rows = []

for row in section_requirements_df.itertuples(index=False):

    term_id = int(row.term_id)
    section_id = int(row.section_id)
    requirement_id = int(row.requirement_id)

    required_count = int(row.sessions_per_week)

    existing_count = len(
        solver_existing_allocations[
            (solver_existing_allocations["term_id"].astype(int) == term_id)
            & (solver_existing_allocations["section_id"].astype(int) == section_id)
            & (solver_existing_allocations["requirement_id"].astype(int) == requirement_id)
        ]
    )

    remaining_count = max(required_count - existing_count, 0)

    for i in range(remaining_count):

        instance_number = existing_count + i + 1

        weekly_session_rows.append({
            "session_key": f"S{section_id}_R{requirement_id}_W{instance_number}",
            "term_id": term_id,
            "section_id": section_id,
            "section_code": row.section_code,
            "requirement_id": requirement_id,
            "course_id": int(row.course_id),
            "kind": row.kind,
            "instance_number": instance_number,
            "duration_minutes": int(row.duration_minutes),
            "required_room_kind": row.required_room_kind
        })


weekly_sessions = pd.DataFrame(weekly_session_rows)

print("Sessions still needing allocation:", len(weekly_sessions))

display(weekly_sessions.head(20))

Sessions still needing allocation: 24


,session_key,term_id,section_id,section_code,requirement_id,course_id,kind,instance_number,duration_minutes,required_room_kind
0,S2_R1_W1,1,2,AI301-S2,1,1,PRACTICAL,1,120,LAB
1,S3_R1_W1,1,3,AI301-S3,1,1,PRACTICAL,1,120,LAB
2,S4_R1_W1,1,4,AI301-S4,1,1,PRACTICAL,1,120,LAB
3,S5_R1_W1,1,5,AI301-S5,1,1,PRACTICAL,1,120,LAB
4,S6_R2_W1,1,6,AI302-S1,2,2,PRACTICAL,1,120,LAB
5,S7_R2_W1,1,7,AI302-S2,2,2,PRACTICAL,1,120,LAB
6,S8_R2_W1,1,8,AI302-S3,2,2,PRACTICAL,1,120,LAB
7,S9_R2_W1,1,9,AI302-S4,2,2,PRACTICAL,1,120,LAB
8,S10_R2_W1,1,10,AI302-S5,2,2,PRACTICAL,1,120,LAB
9,S11_R3_W1,1,11,AI303-S1,3,3,PRACTICAL,1,120,LAB


In [22]:
# -------------------------
# Strict Candidate Checker
# -------------------------

def check_candidate_strict(
    term_id,
    section_id,
    requirement_id,
    instructor_id,
    room_id,
    weekday,
    starts_at,
    ends_at,
    existing,
    session_date=None
):
    result = check_allocation(
        term_id=term_id,
        section_id=section_id,
        requirement_id=requirement_id,
        instructor_id=instructor_id,
        room_id=room_id,
        weekday=weekday,
        starts_at=starts_at,
        ends_at=ends_at,
        existing=existing,
        session_date=session_date
    )

    reasons = set(result["reasons"])

    start = minutes(starts_at)
    end = minutes(ends_at)

    # Check conflicts with an existing allocation
    # belonging to the same section
    for old in existing.itertuples(index=False):

        if int(old.term_id) != int(term_id):
            continue

        if int(old.weekday) != int(weekday):
            continue

        if int(old.section_id) != int(section_id):
            continue

        if not overlaps(
            start,
            end,
            minutes(old.starts_at),
            minutes(old.ends_at)
        ):
            continue

        reasons.add("GROUP_CONFLICT")

        if int(old.room_id) == int(room_id):
            reasons.add("ROOM_CONFLICT")

        if int(old.instructor_id) == int(instructor_id):
            reasons.add("STAFF_CONFLICT")

    ordered = [
        reason
        for reason in PRIORITY
        if reason in reasons
    ]

    return {
        "primary_result":
            ordered[0] if ordered else "FEASIBLE",

        "reasons": ordered,

        "slot_id": result["slot_id"]
    }


print("Strict candidate checker ready ")

Strict candidate checker ready 


In [23]:
# -------------------------
# Soft Scoring Definitions
# -------------------------

SOFT_SCORE_DEFINITIONS = {

    "STAFF_PREFERENCE": {
        "max_points": 30,
        "scope": "CANDIDATE",
        "description":
            "Prefer slots explicitly marked as PREFERRED "
            "by the assigned instructor"
    },

    "CAPACITY_FIT": {
        "max_points": 25,
        "scope": "CANDIDATE",
        "description":
            "Prefer rooms that fit the student count "
            "with less unused capacity"
    },

    "EQUIPMENT_MATCH": {
        "max_points": 20,
        "scope": "CANDIDATE",
        "description":
            "Prefer rooms whose available equipment "
            "closely matches the session requirements"
    },

    "COMPACTNESS": {
        "points_per_adjacent_pair": 15,
        "scope": "SCHEDULE",
        "description":
            "Reward adjacent sessions for the same section "
            "or instructor to reduce unnecessary timetable gaps"
    },

    "ROOM_UTILIZATION": {
        "penalty_per_active_room": 10,
        "scope": "SCHEDULE",
        "description":
            "Reduce unnecessary room spreading by preferring "
            "solutions that use fewer active rooms when feasible"
    }
}


SOFT_SCORE_DEFINITIONS

{'STAFF_PREFERENCE': {'max_points': 30,
  'scope': 'CANDIDATE',
  'description': 'Prefer slots explicitly marked as PREFERRED by the assigned instructor'},
 'CAPACITY_FIT': {'max_points': 25,
  'scope': 'CANDIDATE',
  'description': 'Prefer rooms that fit the student count with less unused capacity'},
 'EQUIPMENT_MATCH': {'max_points': 20,
  'scope': 'CANDIDATE',
  'description': 'Prefer rooms whose available equipment closely matches the session requirements'},
 'COMPACTNESS': {'points_per_adjacent_pair': 15,
  'scope': 'SCHEDULE',
  'description': 'Reward adjacent sessions for the same section or instructor to reduce unnecessary timetable gaps'},
 'ROOM_UTILIZATION': {'penalty_per_active_room': 10,
  'scope': 'SCHEDULE',
  'description': 'Reduce unnecessary room spreading by preferring solutions that use fewer active rooms when feasible'}}

In [24]:
print("Required weekly sessions:", section_requirements_df["sessions_per_week"].sum())
print("Already allocated:", len(solver_existing_allocations))
print("Still to schedule:", len(weekly_sessions))

Required weekly sessions: 25
Already allocated: 1
Still to schedule: 24


In [ ]:
def generate_all_candidates(weekly_sessions):

    candidate_rows = []
    diagnostic_rows = []

    for session in weekly_sessions.itertuples(
        index=False
    ):

        term_id = int(session.term_id)
        section_id = int(session.section_id)
        requirement_id = int(
            session.requirement_id
        )

        # -------------------------
        # Eligible instructors only
        # -------------------------

        instructor_ids = sorted({
            instructor_id
            for (
                s,
                r,
                instructor_id
            ) in eligible
            if s == section_id
            and r == requirement_id
        })

        reason_counter = Counter()

        if not instructor_ids:

            diagnostic_rows.append({
                "session_key":
                    session.session_key,

                "reason_code":
                    "INELIGIBLE_INSTRUCTOR",

                "count": 1
            })

            continue

        # -------------------------
        # Try every instructor
        # -------------------------

        for instructor_id in instructor_ids:

            submission = (
                submission_by_staff.get(
                    (
                        term_id,
                        instructor_id
                    )
                )
            )

            # -------------------------
            # Try every time slot
            # -------------------------

            for slot_id, slot in slots.items():

                if int(slot["term_id"]) != term_id:
                    continue

                # -------------------------
                # Try every room
                # -------------------------

                for room_id, room in rooms.items():

                    result = check_candidate_strict(
                        term_id=term_id,
                        section_id=section_id,
                        requirement_id=requirement_id,
                        instructor_id=instructor_id,
                        room_id=room_id,
                        weekday=slot["weekday"],
                        starts_at=slot["starts_at"],
                        ends_at=slot["ends_at"],
                        existing=solver_existing_allocations
                    )

                    # -------------------------
                    # Hard conflict
                    # -------------------------

                    if (
                        result["primary_result"]
                        != "FEASIBLE"
                    ):

                        for reason in result["reasons"]:
                            reason_counter[reason] += 1

                        continue

                    # =================================================
                    # SOFT SCORING
                    # =================================================

                    # -------------------------
                    # 1) Staff Preference
                    # Max = 30
                    # -------------------------

                    if submission is not None:

                        availability_kind = availability.get(
                            (
                                int(submission.id),
                                int(slot_id)
                            ),
                            "UNKNOWN"
                        )

                    else:

                        availability_kind = "UNKNOWN"

                    staff_preference_score = (
                        30
                        if availability_kind == "PREFERRED"
                        else 0
                    )

                    # -------------------------
                    # 2) Capacity Fit
                    # Max = 25
                    # -------------------------

                    student_count = int(
                        students_by_section.get(
                            section_id,
                            0
                        )
                    )

                    room_capacity = int(
                        room["capacity"]
                    )

                    if room_capacity > 0:

                        capacity_fit_pct = round(
                            (
                                student_count
                                / room_capacity
                            )
                            * 100
                        )

                    else:

                        capacity_fit_pct = 0

                    capacity_fit_pct = max(
                        0,
                        min(
                            capacity_fit_pct,
                            100
                        )
                    )

                    capacity_fit_score = round(
                        25
                        * capacity_fit_pct
                        / 100
                    )

                    # -------------------------
                    # 3) Equipment Match
                    # Max = 20
                    # -------------------------

                    required_equipment_items = [
                        (
                            int(equipment_id),
                            int(quantity)
                        )

                        for (
                            req_id,
                            equipment_id
                        ), quantity in required.items()

                        if int(req_id)
                        == requirement_id
                    ]

                    if not required_equipment_items:

                        equipment_fit_pct = 100

                    else:

                        equipment_ratios = []

                        for (
                            equipment_id,
                            required_quantity
                        ) in required_equipment_items:

                            available_quantity = int(
                                available.get(
                                    (
                                        room_id,
                                        equipment_id
                                    ),
                                    0
                                )
                            )

                            if available_quantity <= 0:

                                fit_ratio = 0

                            else:

                                fit_ratio = min(
                                    required_quantity
                                    / available_quantity,
                                    1.0
                                )

                            equipment_ratios.append(
                                fit_ratio
                            )

                        equipment_fit_pct = round(
                            (
                                sum(
                                    equipment_ratios
                                )
                                / len(
                                    equipment_ratios
                                )
                            )
                            * 100
                        )

                    equipment_fit_pct = max(
                        0,
                        min(
                            equipment_fit_pct,
                            100
                        )
                    )

                    equipment_match_score = round(
                        20
                        * equipment_fit_pct
                        / 100
                    )

                    # -------------------------
                    # Candidate-level score
                    # Maximum = 75
                    # -------------------------

                    total_score = (
                        staff_preference_score
                        + capacity_fit_score
                        + equipment_match_score
                    )

                    soft_reason_codes = [
                        "CAPACITY_FIT",
                        "EQUIPMENT_MATCH"
                    ]

                    if (
                        availability_kind
                        == "PREFERRED"
                    ):

                        soft_reason_codes.insert(
                            0,
                            "STAFF_PREFERENCE"
                        )

                    # -------------------------
                    # Store candidate
                    # -------------------------

                    candidate_rows.append({

                        "session_key":
                            session.session_key,

                        "term_id":
                            term_id,

                        "section_id":
                            section_id,

                        "section_code":
                            session.section_code,

                        "requirement_id":
                            requirement_id,

                        "instance_number":
                            int(
                                session.instance_number
                            ),

                        "instructor_id":
                            int(instructor_id),

                        "room_id":
                            int(room_id),

                        "slot_id":
                            int(slot_id),

                        "weekday":
                            int(
                                slot["weekday"]
                            ),

                        "starts_at":
                            slot["starts_at"],

                        "ends_at":
                            slot["ends_at"],

                        "availability":
                            availability_kind,

                        "student_count":
                            student_count,

                        "room_capacity":
                            room_capacity,

                        # Score breakdown

                        "staff_preference_score":
                            int(
                                staff_preference_score
                            ),

                        "capacity_fit_pct":
                            int(
                                capacity_fit_pct
                            ),

                        "capacity_fit_score":
                            int(
                                capacity_fit_score
                            ),

                        "equipment_fit_pct":
                            int(
                                equipment_fit_pct
                            ),

                        "equipment_match_score":
                            int(
                                equipment_match_score
                            ),

                        "total_score":
                            int(
                                total_score
                            ),

                        "soft_reason_codes":
                            soft_reason_codes
                    })

        # -------------------------
        # Save rejected candidate reasons
        # -------------------------

        for reason, count in reason_counter.items():

            diagnostic_rows.append({
                "session_key":
                    session.session_key,

                "reason_code":
                    reason,

                "count":
                    int(count)
            })

    # -------------------------
    # Build DataFrames
    # -------------------------

    candidates = pd.DataFrame(
        candidate_rows
    )

    diagnostics = pd.DataFrame(
        diagnostic_rows
    )

    if not candidates.empty:

        candidates = candidates.reset_index(
            drop=True
        )

        candidates[
            "candidate_id"
        ] = candidates.index.astype(int)

    return candidates, diagnostics


print("Candidate generator ready ")

Candidate generator ready 


In [26]:
candidates, candidate_diagnostics = (
    generate_all_candidates(
        weekly_sessions
    )
)

print(
    "Total feasible candidates:",
    len(candidates)
)

display(
    candidates.head(20)
)

Total feasible candidates: 4191


,session_key,term_id,section_id,section_code,requirement_id,instance_number,instructor_id,room_id,slot_id,weekday,...,student_count,room_capacity,staff_preference_score,capacity_fit_pct,capacity_fit_score,equipment_fit_pct,equipment_match_score,total_score,soft_reason_codes,candidate_id
0,S2_R1_W1,1,2,AI301-S2,1,1,4,1,1,6,...,40,44,0,91,23,91,18,41,"[CAPACITY_FIT, EQUIPMENT_MATCH]",0
1,S2_R1_W1,1,2,AI301-S2,1,1,4,3,1,6,...,40,44,0,91,23,91,18,41,"[CAPACITY_FIT, EQUIPMENT_MATCH]",1
2,S2_R1_W1,1,2,AI301-S2,1,1,4,4,1,6,...,40,44,0,91,23,91,18,41,"[CAPACITY_FIT, EQUIPMENT_MATCH]",2
3,S2_R1_W1,1,2,AI301-S2,1,1,4,5,1,6,...,40,44,0,91,23,91,18,41,"[CAPACITY_FIT, EQUIPMENT_MATCH]",3
4,S2_R1_W1,1,2,AI301-S2,1,1,4,6,1,6,...,40,44,0,91,23,91,18,41,"[CAPACITY_FIT, EQUIPMENT_MATCH]",4
5,S2_R1_W1,1,2,AI301-S2,1,1,4,7,1,6,...,40,44,0,91,23,91,18,41,"[CAPACITY_FIT, EQUIPMENT_MATCH]",5
6,S2_R1_W1,1,2,AI301-S2,1,1,4,8,1,6,...,40,44,0,91,23,91,18,41,"[CAPACITY_FIT, EQUIPMENT_MATCH]",6
7,S2_R1_W1,1,2,AI301-S2,1,1,4,9,1,6,...,40,44,0,91,23,91,18,41,"[CAPACITY_FIT, EQUIPMENT_MATCH]",7
8,S2_R1_W1,1,2,AI301-S2,1,1,4,1,2,6,...,40,44,0,91,23,91,18,41,"[CAPACITY_FIT, EQUIPMENT_MATCH]",8
9,S2_R1_W1,1,2,AI301-S2,1,1,4,3,2,6,...,40,44,0,91,23,91,18,41,"[CAPACITY_FIT, EQUIPMENT_MATCH]",9


In [27]:
candidate_counts = (
    candidates
    .groupby("session_key")
    .size()
    .rename("candidate_count")
    .reset_index()
)

coverage = (
    weekly_sessions[
        [
            "session_key",
            "section_code",
            "kind"
        ]
    ]
    .merge(
        candidate_counts,
        on="session_key",
        how="left"
    )
)

coverage["candidate_count"] = (
    coverage["candidate_count"]
    .fillna(0)
    .astype(int)
)

display(coverage)

,session_key,section_code,kind,candidate_count
0,S2_R1_W1,AI301-S2,PRACTICAL,303
1,S3_R1_W1,AI301-S3,PRACTICAL,159
2,S4_R1_W1,AI301-S4,PRACTICAL,159
3,S5_R1_W1,AI301-S5,PRACTICAL,159
4,S6_R2_W1,AI302-S1,PRACTICAL,159
5,S7_R2_W1,AI302-S2,PRACTICAL,159
6,S8_R2_W1,AI302-S3,PRACTICAL,159
7,S9_R2_W1,AI302-S4,PRACTICAL,159
8,S10_R2_W1,AI302-S5,PRACTICAL,159
9,S11_R3_W1,AI303-S1,PRACTICAL,159


In [28]:
problem_sessions = coverage[
    coverage["candidate_count"] == 0
]

print(
    "Sessions with zero candidates:",
    len(problem_sessions)
)

if not problem_sessions.empty:
    display(problem_sessions)

    display(
        candidate_diagnostics[
            candidate_diagnostics[
                "session_key"
            ].isin(
                problem_sessions[
                    "session_key"
                ]
            )
        ]
        .sort_values(
            ["session_key", "count"],
            ascending=[True, False]
        )
    )
else:
    print(
        "Every weekly session has at least one feasible candidate ✅"
    )

Sessions with zero candidates: 1


,session_key,section_code,kind,candidate_count
13,S15_R3_W1,AI303-S5,PRACTICAL,0


,session_key,reason_code,count
54,S15_R3_W1,AVAILABILITY_NOT_CONFIRMED,400
55,S15_R3_W1,EQUIPMENT_SHORTAGE,220
57,S15_R3_W1,ROOM_TYPE_MISMATCH,200
56,S15_R3_W1,CAPACITY_SHORTAGE,20
58,S15_R3_W1,ROOM_CONFLICT,1


In [29]:
UNSCHEDULED_PENALTY = 100000

model = cp_model.CpModel()

# Decision variable for every feasible candidate
x = {}

for row in candidates.itertuples(index=False):
    candidate_id = int(row.candidate_id)

    x[candidate_id] = model.NewBoolVar(
        f"candidate_{candidate_id}"
    )


# One UNSCHEDULED variable for every weekly session
unscheduled = {}

for session_key in weekly_sessions["session_key"]:
    unscheduled[session_key] = model.NewBoolVar(
        f"unscheduled_{session_key}"
    )

print("Decision variables created ")

Decision variables created 


In [30]:
for session in weekly_sessions.itertuples(index=False):

    session_key = session.session_key

    session_candidate_ids = (
        candidates.loc[
            candidates["session_key"] == session_key,
            "candidate_id"
        ]
        .astype(int)
        .tolist()
    )

    model.Add(
        sum(
            x[candidate_id]
            for candidate_id in session_candidate_ids
        )
        + unscheduled[session_key]
        == 1
    )

print("Session assignment constraints added ")

Session assignment constraints added 


In [31]:
for _, group in candidates.groupby(
    ["term_id", "slot_id", "room_id"]
):

    candidate_ids = (
        group["candidate_id"]
        .astype(int)
        .tolist()
    )

    if len(candidate_ids) > 1:
        model.Add(
            sum(
                x[candidate_id]
                for candidate_id in candidate_ids
            )
            <= 1
        )

print("Room conflict constraints added ")

Room conflict constraints added 


In [32]:
for _, group in candidates.groupby(
    ["term_id", "slot_id", "instructor_id"]
):

    candidate_ids = (
        group["candidate_id"]
        .astype(int)
        .tolist()
    )

    if len(candidate_ids) > 1:
        model.Add(
            sum(
                x[candidate_id]
                for candidate_id in candidate_ids
            )
            <= 1
        )

print("Staff conflict constraints added ")

Staff conflict constraints added 


In [33]:
for _, group in candidates.groupby(
    ["term_id", "slot_id", "section_id"]
):

    candidate_ids = (
        group["candidate_id"]
        .astype(int)
        .tolist()
    )

    if len(candidate_ids) > 1:
        model.Add(
            sum(
                x[candidate_id]
                for candidate_id in candidate_ids
            )
            <= 1
        )

print("Section conflict constraints added ")

Section conflict constraints added 


In [34]:
group_candidate_map = defaultdict(list)

for row in candidates.itertuples(index=False):

    section_id = int(row.section_id)

    section_groups = groups_by_section.get(
        section_id,
        set()
    )

    for group_id in section_groups:

        key = (
            int(row.term_id),
            int(row.slot_id),
            int(group_id)
        )

        group_candidate_map[key].append(
            int(row.candidate_id)
        )


for candidate_ids in group_candidate_map.values():

    if len(candidate_ids) > 1:

        model.Add(
            sum(
                x[candidate_id]
                for candidate_id in candidate_ids
            )
            <= 1
        )

print("Student-group conflict constraints added ")

Student-group conflict constraints added 


In [35]:
# =================================================
# GLOBAL SOFT OBJECTIVE
# =================================================


# =================================================
# 1) Candidate-Level Score
#
# Staff Preference
# Capacity Fit
# Equipment Match
# =================================================

candidate_score = sum(
    int(row.total_score)
    * x[int(row.candidate_id)]

    for row in candidates.itertuples(
        index=False
    )
)


# =================================================
# 2) Compactness
#
# Memory-efficient version:
# instead of comparing every candidate with every
# possible adjacent candidate, we create occupancy
# variables per section/instructor + slot.
# =================================================

COMPACTNESS_POINTS = 15


# -------------------------
# Find Adjacent Time Slots
# -------------------------

slot_rows = []

for slot_id, slot in slots.items():

    slot_rows.append({
        "slot_id":
            int(slot_id),

        "term_id":
            int(slot["term_id"]),

        "weekday":
            int(slot["weekday"]),

        "start_minute":
            minutes(
                slot["starts_at"]
            ),

        "end_minute":
            minutes(
                slot["ends_at"]
            )
    })


slot_df = pd.DataFrame(
    slot_rows
)


adjacent_slots = []


for (
    term_id,
    weekday
), day_slots in slot_df.groupby(
    [
        "term_id",
        "weekday"
    ]
):

    day_slots = (
        day_slots
        .sort_values(
            "start_minute"
        )
        .reset_index(
            drop=True
        )
    )

    for i in range(
        len(day_slots) - 1
    ):

        current_slot = (
            day_slots.iloc[i]
        )

        next_slot = (
            day_slots.iloc[i + 1]
        )

        # Example:
        # 09:00-11:00
        # followed directly by
        # 11:00-13:00
        if (
            int(
                current_slot[
                    "end_minute"
                ]
            )
            ==
            int(
                next_slot[
                    "start_minute"
                ]
            )
        ):

            adjacent_slots.append(
                (
                    int(term_id),
                    int(weekday),
                    int(
                        current_slot[
                            "slot_id"
                        ]
                    ),
                    int(
                        next_slot[
                            "slot_id"
                        ]
                    )
                )
            )


print(
    "Adjacent slot pairs:",
    len(adjacent_slots)
)


# =================================================
# Section Occupancy
#
# section_occupancy = 1 when a section has
# a selected session in a specific slot.
# =================================================

section_occupancy = {}


for keys, group in candidates.groupby(
    [
        "term_id",
        "weekday",
        "section_id",
        "slot_id"
    ]
):

    (
        term_id,
        weekday,
        section_id,
        slot_id
    ) = map(
        int,
        keys
    )

    candidate_ids = (
        group[
            "candidate_id"
        ]
        .astype(int)
        .tolist()
    )

    occupancy_var = (
        model.NewBoolVar(
            f"section_occ_"
            f"{term_id}_"
            f"{weekday}_"
            f"{section_id}_"
            f"{slot_id}"
        )
    )

    # Section conflict constraints already
    # guarantee that this sum <= 1.
    model.Add(
        occupancy_var
        ==
        sum(
            x[candidate_id]
            for candidate_id
            in candidate_ids
        )
    )

    section_occupancy[
        (
            term_id,
            weekday,
            section_id,
            slot_id
        )
    ] = occupancy_var


print(
    "Section occupancy variables:",
    len(section_occupancy)
)


# =================================================
# Instructor Occupancy
#
# instructor_occupancy = 1 when an instructor
# has a selected session in a specific slot.
# =================================================

instructor_occupancy = {}


for keys, group in candidates.groupby(
    [
        "term_id",
        "weekday",
        "instructor_id",
        "slot_id"
    ]
):

    (
        term_id,
        weekday,
        instructor_id,
        slot_id
    ) = map(
        int,
        keys
    )

    candidate_ids = (
        group[
            "candidate_id"
        ]
        .astype(int)
        .tolist()
    )

    occupancy_var = (
        model.NewBoolVar(
            f"instructor_occ_"
            f"{term_id}_"
            f"{weekday}_"
            f"{instructor_id}_"
            f"{slot_id}"
        )
    )

    # Staff conflict constraints already
    # guarantee that this sum <= 1.
    model.Add(
        occupancy_var
        ==
        sum(
            x[candidate_id]
            for candidate_id
            in candidate_ids
        )
    )

    instructor_occupancy[
        (
            term_id,
            weekday,
            instructor_id,
            slot_id
        )
    ] = occupancy_var


print(
    "Instructor occupancy variables:",
    len(instructor_occupancy)
)


# =================================================
# Compactness Variables
# =================================================

compactness_vars = []


# -------------------------
# A) Same Section
# -------------------------

section_ids = sorted(
    candidates[
        "section_id"
    ]
    .astype(int)
    .unique()
)


for (
    term_id,
    weekday,
    slot_a,
    slot_b
) in adjacent_slots:

    for section_id in section_ids:

        key_a = (
            term_id,
            weekday,
            section_id,
            slot_a
        )

        key_b = (
            term_id,
            weekday,
            section_id,
            slot_b
        )

        if (
            key_a not in section_occupancy
            or
            key_b not in section_occupancy
        ):
            continue

        pair_var = (
            model.NewBoolVar(
                f"section_compact_"
                f"{term_id}_"
                f"{weekday}_"
                f"{section_id}_"
                f"{slot_a}_"
                f"{slot_b}"
            )
        )

        # pair_var = 1 only when
        # both adjacent slots are selected

        model.Add(
            pair_var
            <= section_occupancy[
                key_a
            ]
        )

        model.Add(
            pair_var
            <= section_occupancy[
                key_b
            ]
        )

        model.Add(
            pair_var
            >=
            section_occupancy[
                key_a
            ]
            +
            section_occupancy[
                key_b
            ]
            - 1
        )

        compactness_vars.append(
            pair_var
        )


# -------------------------
# B) Same Instructor
# -------------------------

instructor_ids = sorted(
    candidates[
        "instructor_id"
    ]
    .astype(int)
    .unique()
)


for (
    term_id,
    weekday,
    slot_a,
    slot_b
) in adjacent_slots:

    for instructor_id in instructor_ids:

        key_a = (
            term_id,
            weekday,
            instructor_id,
            slot_a
        )

        key_b = (
            term_id,
            weekday,
            instructor_id,
            slot_b
        )

        if (
            key_a not in instructor_occupancy
            or
            key_b not in instructor_occupancy
        ):
            continue

        pair_var = (
            model.NewBoolVar(
                f"instructor_compact_"
                f"{term_id}_"
                f"{weekday}_"
                f"{instructor_id}_"
                f"{slot_a}_"
                f"{slot_b}"
            )
        )

        model.Add(
            pair_var
            <= instructor_occupancy[
                key_a
            ]
        )

        model.Add(
            pair_var
            <= instructor_occupancy[
                key_b
            ]
        )

        model.Add(
            pair_var
            >=
            instructor_occupancy[
                key_a
            ]
            +
            instructor_occupancy[
                key_b
            ]
            - 1
        )

        compactness_vars.append(
            pair_var
        )


compactness_bonus = (
    COMPACTNESS_POINTS
    * sum(
        compactness_vars
    )
)


print(
    "Compactness variables:",
    len(compactness_vars)
)


# =================================================
# 3) Room Utilization
#
# Prefer using fewer active rooms when all other
# conditions are similar.
# =================================================

ROOM_ACTIVE_PENALTY = 10

room_used = {}


for room_id, group in candidates.groupby(
    "room_id"
):

    room_id = int(
        room_id
    )

    room_candidate_ids = (
        group[
            "candidate_id"
        ]
        .astype(int)
        .tolist()
    )

    room_used[
        room_id
    ] = model.NewBoolVar(
        f"room_used_{room_id}"
    )

    # Any selected candidate using this room
    # forces room_used = 1.
    for candidate_id in room_candidate_ids:

        model.Add(
            x[candidate_id]
            <= room_used[
                room_id
            ]
        )

    # room_used cannot be 1 if no selected
    # candidate uses the room.
    model.Add(
        room_used[
            room_id
        ]
        <=
        sum(
            x[candidate_id]
            for candidate_id
            in room_candidate_ids
        )
    )


room_utilization_penalty = (
    ROOM_ACTIVE_PENALTY
    * sum(
        room_used.values()
    )
)


print(
    "Room utilization variables:",
    len(room_used)
)


# =================================================
# 4) Unscheduled Penalty
# =================================================

unscheduled_cost = sum(
    UNSCHEDULED_PENALTY
    * unscheduled[
        session_key
    ]

    for session_key
    in unscheduled
)


# =================================================
# 5) FINAL OBJECTIVE
# =================================================

model.Maximize(
    candidate_score
    + compactness_bonus
    - room_utilization_penalty
    - unscheduled_cost
)


print(
    "Global objective created "
)

Adjacent slot pairs: 15
Section occupancy variables: 458
Instructor occupancy variables: 278
Compactness variables: 551
Room utilization variables: 18
Global objective created 


In [36]:
solver = cp_model.CpSolver()

solver.parameters.max_time_in_seconds = 30
solver.parameters.num_search_workers = 4

status = solver.Solve(model)

print(
    "Solver status:",
    solver.StatusName(status)
)

Solver status: FEASIBLE


In [37]:
selected_candidate_ids = [
    candidate_id
    for candidate_id, variable in x.items()
    if solver.Value(variable) == 1
]

selected_schedule = (
    candidates[
        candidates["candidate_id"].isin(
            selected_candidate_ids
        )
    ]
    .sort_values(
        [
            "weekday",
            "starts_at",
            "section_code"
        ]
    )
    .reset_index(drop=True)
)


unscheduled_sessions = [
    session_key
    for session_key, variable
    in unscheduled.items()
    if solver.Value(variable) == 1
]


print(
    "Scheduled sessions:",
    len(selected_schedule)
)

print(
    "Unscheduled sessions:",
    len(unscheduled_sessions)
)

print(
    "Unscheduled keys:",
    unscheduled_sessions
)

display(selected_schedule)

Scheduled sessions: 23
Unscheduled sessions: 1
Unscheduled keys: ['S15_R3_W1']


,session_key,term_id,section_id,section_code,requirement_id,instance_number,instructor_id,room_id,slot_id,weekday,...,student_count,room_capacity,staff_preference_score,capacity_fit_pct,capacity_fit_score,equipment_fit_pct,equipment_match_score,total_score,soft_reason_codes,candidate_id
0,S25_R5_W1,1,25,AI305-S5,5,1,13,11,14,2,...,38,60,0,63,16,100,20,36,"[CAPACITY_FIT, EQUIPMENT_MATCH]",4121
1,S10_R2_W1,1,10,AI302-S5,2,1,13,1,15,2,...,38,44,0,86,22,91,18,40,"[CAPACITY_FIT, EQUIPMENT_MATCH]",1527
2,S20_R4_W1,1,20,AI304-S5,4,1,8,11,17,3,...,40,60,0,67,17,100,20,37,"[CAPACITY_FIT, EQUIPMENT_MATCH]",3151
3,S5_R1_W1,1,5,AI301-S5,1,1,8,3,18,3,...,40,44,0,91,23,91,18,41,"[CAPACITY_FIT, EQUIPMENT_MATCH]",757
4,S11_R3_W1,1,11,AI303-S1,3,1,14,1,19,3,...,40,44,0,91,23,91,18,41,"[CAPACITY_FIT, EQUIPMENT_MATCH]",1718
5,S12_R3_W1,1,12,AI303-S2,3,1,15,3,19,3,...,36,44,0,82,20,91,18,38,"[CAPACITY_FIT, EQUIPMENT_MATCH]",1878
6,S13_R3_W1,1,13,AI303-S3,3,1,16,1,20,3,...,38,44,0,86,22,91,18,40,"[CAPACITY_FIT, EQUIPMENT_MATCH]",2044
7,S14_R3_W1,1,14,AI303-S4,3,1,17,3,20,3,...,40,44,0,91,23,91,18,41,"[CAPACITY_FIT, EQUIPMENT_MATCH]",2204
8,S17_R4_W1,1,17,AI304-S2,4,1,5,11,20,3,...,40,60,0,67,17,100,20,37,"[CAPACITY_FIT, EQUIPMENT_MATCH]",2581
9,S23_R5_W1,1,23,AI305-S3,5,1,11,11,1,6,...,40,60,0,67,17,100,20,37,"[CAPACITY_FIT, EQUIPMENT_MATCH]",3591


In [38]:
def get_session_reason_codes(session_key):
    session_diag = candidate_diagnostics[
        candidate_diagnostics["session_key"] == session_key
    ]

    if session_diag.empty:
        return ["NO_FEASIBLE_CANDIDATE"]

    present_reasons = set(
        session_diag["reason_code"].astype(str)
    )

    ordered = [
        reason
        for reason in PRIORITY
        if reason in present_reasons
    ]

    extra = sorted(
        present_reasons - set(ordered)
    )

    return ordered + extra


unscheduled_rows = []

for session_key in unscheduled_sessions:

    session = weekly_sessions[
        weekly_sessions["session_key"] == session_key
    ].iloc[0]

    diagnostics = candidate_diagnostics[
        candidate_diagnostics["session_key"] == session_key
    ]

    diagnostic_counts = {
        row.reason_code: int(row.count)
        for row in diagnostics.itertuples(index=False)
    }

    unscheduled_rows.append({
        "session_key": session_key,
        "section_id": int(session["section_id"]),
        "section_code": session["section_code"],
        "requirement_id": int(session["requirement_id"]),
        "kind": session["kind"],
        "status": "UNSCHEDULED",
        "reason_codes": get_session_reason_codes(
            session_key
        ),
        "diagnostic_counts": diagnostic_counts
    })


unscheduled_report = pd.DataFrame(
    unscheduled_rows
)

display(unscheduled_report)

,session_key,section_id,section_code,requirement_id,kind,status,reason_codes,diagnostic_counts
0,S15_R3_W1,15,AI303-S5,3,PRACTICAL,UNSCHEDULED,"[AVAILABILITY_NOT_CONFIRMED, ROOM_CONFLICT, RO...","{'AVAILABILITY_NOT_CONFIRMED': 400, 'EQUIPMENT..."


In [39]:
def candidate_conflicts_with_schedule(
    candidate,
    selected_others
):
    candidate_groups = groups_by_section.get(
        int(candidate["section_id"]),
        set()
    )

    for other in selected_others.itertuples(
        index=False
    ):

        if int(other.term_id) != int(
            candidate["term_id"]
        ):
            continue

        if int(other.weekday) != int(
            candidate["weekday"]
        ):
            continue

        if not overlaps(
            minutes(candidate["starts_at"]),
            minutes(candidate["ends_at"]),
            minutes(other.starts_at),
            minutes(other.ends_at)
        ):
            continue

        # Same room
        if int(other.room_id) == int(
            candidate["room_id"]
        ):
            return True

        # Same instructor
        if int(other.instructor_id) == int(
            candidate["instructor_id"]
        ):
            return True

        # Same section
        if int(other.section_id) == int(
            candidate["section_id"]
        ):
            return True

        # Shared student groups
        other_groups = groups_by_section.get(
            int(other.section_id),
            set()
        )

        if candidate_groups & other_groups:
            return True

    return False

In [40]:
def get_ranked_alternatives(
    session_key,
    limit=5
):

    chosen = selected_schedule[
        selected_schedule["session_key"] == session_key
    ]

    if chosen.empty:
        return []

    selected_candidate_id = int(
        chosen.iloc[0]["candidate_id"]
    )

    selected_others = selected_schedule[
        selected_schedule["session_key"] != session_key
    ]

    pool = candidates[
        (candidates["session_key"] == session_key)
        &
        (candidates["candidate_id"] != selected_candidate_id)
    ].copy()

    # OLD RANKING LOGIC
    pool["preference_rank"] = (
        pool["availability"]
        .apply(
            lambda x: 0 if x == "PREFERRED" else 1
        )
    )

    pool = pool.sort_values(
        [
            "preference_rank",
            "weekday",
            "starts_at",
            "room_id"
        ],
        ascending=[
            True,
            True,
            True,
            True
        ]
    )

    alternatives = []

    for _, candidate in pool.iterrows():

        if candidate_conflicts_with_schedule(
            candidate,
            selected_others
        ):
            continue

        alternatives.append({

            "candidate_id": int(
                candidate["candidate_id"]
            ),

            "instructor_id": int(
                candidate["instructor_id"]
            ),

            "room_id": int(
                candidate["room_id"]
            ),

            "slot_id": int(
                candidate["slot_id"]
            ),

            "weekday": int(
                candidate["weekday"]
            ),

            "starts_at":
                candidate["starts_at"],

            "ends_at":
                candidate["ends_at"],

            "availability":
                candidate["availability"]
        })

        if len(alternatives) >= limit:
            break

    return alternatives

In [41]:
alternatives = get_ranked_alternatives(
    "S2_R1_W1"
)

print(
    "Number of alternatives:",
    len(alternatives)
)

for i, alt in enumerate(
    alternatives,
    start=1
):
    print(
        i,
        "Room:", alt["room_id"],
        "| Instructor:", alt["instructor_id"],
        "| Slot:", alt["slot_id"],
        "| Availability:", alt["availability"]
    )

Number of alternatives: 5
1 Room: 1 | Instructor: 5 | Slot: 9 | Availability: AVAILABLE
2 Room: 3 | Instructor: 5 | Slot: 9 | Availability: AVAILABLE
3 Room: 4 | Instructor: 5 | Slot: 9 | Availability: AVAILABLE
4 Room: 5 | Instructor: 5 | Slot: 9 | Availability: AVAILABLE
5 Room: 6 | Instructor: 5 | Slot: 9 | Availability: AVAILABLE


In [44]:
def calculate_room_score(
    room_id,
    section_id,
    requirement_id
):
    room = rooms[int(room_id)]

    student_count = students_by_section.get(
        int(section_id),
        0
    )

    room_capacity = int(room["capacity"])

    # 1) Capacity Fit - 50
    if room_capacity < student_count:
        capacity_score = 0
    else:
        capacity_ratio = student_count / room_capacity
        capacity_score = 50 * capacity_ratio

    # 2) Equipment Match - 30
    required_items = [
        (equipment_id, quantity)
        for (req_id, equipment_id), quantity in required.items()
        if int(req_id) == int(requirement_id)
    ]

    if not required_items:
        equipment_score = 30
    else:
        matched = 0

        for equipment_id, required_quantity in required_items:

            available_quantity = available.get(
                (
                    int(room_id),
                    int(equipment_id)
                ),
                0
            )

            if available_quantity >= required_quantity:
                matched += 1

        equipment_score = (
            30 * matched / len(required_items)
        )

    # 3) Room Type - 20
    requirement = requirements[int(requirement_id)]

    if room["kind"] == requirement["required_room_kind"]:
        room_type_score = 20
    else:
        room_type_score = 0

    # Final Score
    room_score = (
        capacity_score
        + equipment_score
        + room_type_score
    )

    return round(room_score, 2)
print(
    calculate_room_score(
        room_id=1,
        section_id=2,
        requirement_id=1
    )
)

95.45


In [46]:
candidates["room_score"] = candidates.apply(
    lambda row: calculate_room_score(
        room_id=row["room_id"],
        section_id=row["section_id"],
        requirement_id=row["requirement_id"]
    ),
    axis=1
)
display(
    candidates[
        [
            "session_key",
            "room_id",
            "instructor_id",
            "slot_id",
            "room_score"
        ]
    ]
    .sort_values(
        "room_score",
        ascending=False
    )
    .head(20)
)

,session_key,room_id,instructor_id,slot_id,room_score
2210,S14_R3_W1,9,17,20,95.45
2209,S14_R3_W1,8,17,20,95.45
2208,S14_R3_W1,7,17,20,95.45
31,S2_R1_W1,9,4,4,95.45
30,S2_R1_W1,8,4,4,95.45
29,S2_R1_W1,7,4,4,95.45
28,S2_R1_W1,6,4,4,95.45
27,S2_R1_W1,5,4,4,95.45
26,S2_R1_W1,4,4,4,95.45
25,S2_R1_W1,3,4,4,95.45


In [ ]:
get_ranked_alternatives(
    "S2_R1_W1"
)

[{'candidate_id': 0,
  'instructor_id': 4,
  'room_id': 1,
  'slot_id': 1,
  'weekday': 6,
  'starts_at': '09:00:00',
  'ends_at': '11:00:00',
  'availability': 'AVAILABLE',
  'score': 41,
  'score_breakdown': {'staff_preference': 0,
   'capacity_fit': 23,
   'capacity_fit_pct': 91,
   'equipment_match': 18,
   'equipment_fit_pct': 91},
  'soft_reason_codes': ['CAPACITY_FIT', 'EQUIPMENT_MATCH']},
 {'candidate_id': 1,
  'instructor_id': 4,
  'room_id': 3,
  'slot_id': 1,
  'weekday': 6,
  'starts_at': '09:00:00',
  'ends_at': '11:00:00',
  'availability': 'AVAILABLE',
  'score': 41,
  'score_breakdown': {'staff_preference': 0,
   'capacity_fit': 23,
   'capacity_fit_pct': 91,
   'equipment_match': 18,
   'equipment_fit_pct': 91},
  'soft_reason_codes': ['CAPACITY_FIT', 'EQUIPMENT_MATCH']},
 {'candidate_id': 3,
  'instructor_id': 4,
  'room_id': 5,
  'slot_id': 1,
  'weekday': 6,
  'starts_at': '09:00:00',
  'ends_at': '11:00:00',
  'availability': 'AVAILABLE',
  'score': 41,
  'score_br

In [ ]:
alternatives = get_ranked_alternatives(
    "S2_R1_W1"
)

print("Number of alternatives:", len(alternatives))

for i, alt in enumerate(alternatives, start=1):
    print(
        i,
        "Room:", alt["room_id"],
        "| Instructor:", alt["instructor_id"],
        "| Slot:", alt["slot_id"],
        "| Score:", alt["score"]
    )

Number of alternatives: 5
1 Room: 1 | Instructor: 4 | Slot: 1 | Score: 41
2 Room: 3 | Instructor: 4 | Slot: 1 | Score: 41
3 Room: 5 | Instructor: 4 | Slot: 1 | Score: 41
4 Room: 6 | Instructor: 4 | Slot: 1 | Score: 41
5 Room: 7 | Instructor: 4 | Slot: 1 | Score: 41


In [ ]:
# -------------------------
# Final Objective Metrics
# -------------------------

candidate_score_value = (
    int(selected_schedule["total_score"].sum())
    if not selected_schedule.empty
    else 0
)


selected_compact_pairs = int(
    sum(
        solver.Value(variable)
        for variable in compactness_vars
    )
)


compactness_bonus_value = (
    selected_compact_pairs
    * COMPACTNESS_POINTS
)


active_room_ids = [
    int(room_id)

    for room_id, variable
    in room_used.items()

    if solver.Value(variable) == 1
]


active_room_count = len(
    active_room_ids
)


room_utilization_penalty_value = (
    active_room_count
    * ROOM_ACTIVE_PENALTY
)


unscheduled_penalty_value = (
    len(unscheduled_sessions)
    * UNSCHEDULED_PENALTY
)


final_objective_value = (
    candidate_score_value
    + compactness_bonus_value
    - room_utilization_penalty_value
    - unscheduled_penalty_value
)


print(
    "Candidate score:",
    candidate_score_value
)

print(
    "Compact adjacent pairs:",
    selected_compact_pairs
)

print(
    "Compactness bonus:",
    compactness_bonus_value
)

print(
    "Active rooms:",
    active_room_count
)

print(
    "Room utilization penalty:",
    room_utilization_penalty_value
)

print(
    "Final objective value:",
    final_objective_value
)

Candidate score: 877
Compact adjacent pairs: 9
Compactness bonus: 135
Active rooms: 4
Room utilization penalty: 40
Final objective value: -99028


In [ ]:
scheduled_payload = []


for row in selected_schedule.itertuples(
    index=False
):

    instructor = accounts[
        int(row.instructor_id)
    ]

    room = rooms[
        int(row.room_id)
    ]

    scheduled_payload.append({

        "session_key":
            row.session_key,

        "section_id":
            int(row.section_id),

        "section_code":
            row.section_code,

        "requirement_id":
            int(row.requirement_id),

        "instance_number":
            int(row.instance_number),

        "status":
            "SCHEDULED",

        "instructor": {
            "id":
                int(row.instructor_id),

            "name":
                instructor.get(
                    "full_name"
                )
        },

        "room": {
            "id":
                int(row.room_id),

            "code":
                room.get("code")
        },

        "slot": {
            "id":
                int(row.slot_id),

            "weekday":
                int(row.weekday),

            "starts_at":
                row.starts_at,

            "ends_at":
                row.ends_at
        },

        "availability":
            row.availability,

        # -------------------------
        # Candidate Score
        # -------------------------

        "score":
            int(row.total_score),

        "score_breakdown": {

            "staff_preference": {
                "score":
                    int(
                        row.staff_preference_score
                    ),

                "max_score":
                    30,

                "availability":
                    row.availability
            },

            "capacity_fit": {
                "score":
                    int(
                        row.capacity_fit_score
                    ),

                "max_score":
                    25,

                "fit_percentage":
                    int(
                        row.capacity_fit_pct
                    ),

                "student_count":
                    int(
                        row.student_count
                    ),

                "room_capacity":
                    int(
                        row.room_capacity
                    )
            },

            "equipment_match": {
                "score":
                    int(
                        row.equipment_match_score
                    ),

                "max_score":
                    20,

                "fit_percentage":
                    int(
                        row.equipment_fit_pct
                    )
            },

            "candidate_total": {
                "score":
                    int(
                        row.total_score
                    ),

                "max_score":
                    75
            }
        },

        "soft_reason_codes":
            list(
                row.soft_reason_codes
            ),

        "alternatives":
            get_ranked_alternatives(
                row.session_key,
                limit=5
            )
    })


print(
    "Scheduled payload ready "
)

Scheduled payload ready 


In [ ]:
# -------------------------
# Post-Solve Conflict Validation
# -------------------------

def validate_selected_schedule(
    schedule_df
):
    problems = []

    rows = list(
        schedule_df.itertuples(
            index=False
        )
    )

    for i in range(len(rows)):

        a = rows[i]

        for j in range(
            i + 1,
            len(rows)
        ):
            b = rows[j]

            if int(a.term_id) != int(b.term_id):
                continue

            if int(a.weekday) != int(b.weekday):
                continue

            if not overlaps(
                minutes(a.starts_at),
                minutes(a.ends_at),
                minutes(b.starts_at),
                minutes(b.ends_at)
            ):
                continue

            reasons = []

            if int(a.room_id) == int(b.room_id):
                reasons.append(
                    "ROOM_CONFLICT"
                )

            if (
                int(a.instructor_id)
                == int(b.instructor_id)
            ):
                reasons.append(
                    "STAFF_CONFLICT"
                )

            if (
                int(a.section_id)
                == int(b.section_id)
            ):
                reasons.append(
                    "SECTION_CONFLICT"
                )

            groups_a = (
                groups_by_section.get(
                    int(a.section_id),
                    set()
                )
            )

            groups_b = (
                groups_by_section.get(
                    int(b.section_id),
                    set()
                )
            )

            if groups_a & groups_b:
                reasons.append(
                    "GROUP_CONFLICT"
                )

            if reasons:
                problems.append({
                    "session_a":
                        a.session_key,

                    "session_b":
                        b.session_key,

                    "reasons":
                        reasons
                })

    return problems


post_solve_conflicts = (
    validate_selected_schedule(
        selected_schedule
    )
)


assert not post_solve_conflicts, \
    post_solve_conflicts


print(
    "Post-solve hard conflicts: 0 "
)

Post-solve hard conflicts: 0 


In [ ]:
solver_payload = {
    "solver_status":
        solver.StatusName(status),

    "version_context": {
        "term_id": int(TERM_ID),

        "working_version": (
            None
            if working_version is None
            else {
                "id": int(working_version["id"]),
                "name": working_version["name"],
                "version_number": int(
                    working_version["version_number"]
                ),
                "state": working_version["state"]
            }
        ),

        "published_version": (
            None
            if published_version is None
            else {
                "id": int(published_version["id"]),
                "name": published_version["name"],
                "version_number": int(
                    published_version["version_number"]
                ),
                "state": published_version["state"]
            }
        ),

        "published_schedule_immutable_enforced":
            True,

        "published_version_available":
            published_version is not None
    },

    "summary": {
        "required_weekly_sessions":
            int(
                section_requirements_df[
                    "sessions_per_week"
                ].sum()
            ),

        "existing_allocations":
            int(
                len(
                    solver_existing_allocations
                )
            ),

        "solver_sessions":
            int(
                len(
                    weekly_sessions
                )
            ),

        "scheduled":
            int(
                len(
                    selected_schedule
                )
            ),

        "unscheduled":
            int(
                len(
                    unscheduled_sessions
                )
            )
    },

    "hard_reason_codes":
        PRIORITY,
    "objective_breakdown": {

    "candidate_level_score":
        int(
            candidate_score_value
        ),

    "compactness": {
        "adjacent_pairs":
            int(
                selected_compact_pairs
            ),

        "points_per_pair":
            int(
                COMPACTNESS_POINTS
            ),

        "bonus":
            int(
                compactness_bonus_value
            )
    },

    "room_utilization": {
        "active_room_count":
            int(
                active_room_count
            ),

        "active_room_ids":
            active_room_ids,

        "penalty_per_active_room":
            int(
                ROOM_ACTIVE_PENALTY
            ),

        "penalty":
            int(
                room_utilization_penalty_value
            )
    },

    "unscheduled": {
        "count":
            int(
                len(
                    unscheduled_sessions
                )
            ),

        "penalty_per_session":
            int(
                UNSCHEDULED_PENALTY
            ),

        "total_penalty":
            int(
                unscheduled_penalty_value
            )
    },

    "final_objective_value":
        int(
            final_objective_value
        )
},
    "soft_score_definitions":
        SOFT_SCORE_DEFINITIONS,

    "date_specific_constraints": {
        "mode":
            "CONCRETE_DATE_CHANGE_REQUEST",

        "term_holidays_loaded":
            int(
                len(
                    term_holidays
                )
            ),

        "room_closures_loaded":
            int(
                len(
                    room_closures
                )
            )
    },

    "scheduled_sessions":
        scheduled_payload,

    "unscheduled_sessions":
        unscheduled_rows
}

In [ ]:
print(
    json.dumps(
        solver_payload,
        indent=2,
        ensure_ascii=False
    )[:5000]
)

{
  "solver_status": "FEASIBLE",
  "version_context": {
    "term_id": 1,
    "working_version": {
      "id": 1,
      "name": "Draft 1",
      "version_number": 1,
      "state": "DRAFT"
    },
    "published_version": null,
    "published_schedule_immutable_enforced": true,
    "published_version_available": false
  },
  "summary": {
    "required_weekly_sessions": 25,
    "existing_allocations": 1,
    "solver_sessions": 24,
    "scheduled": 23,
    "unscheduled": 1
  },
  "hard_reason_codes": [
    "NO_WORKING_SLOT",
    "TERM_HOLIDAY",
    "ROOM_CLOSED",
    "INVALID_DURATION",
    "AVAILABILITY_NOT_CONFIRMED",
    "STAFF_UNAVAILABLE",
    "ROOM_CONFLICT",
    "STAFF_CONFLICT",
    "GROUP_CONFLICT",
    "ROOM_TYPE_MISMATCH",
    "CAPACITY_SHORTAGE",
    "EQUIPMENT_SHORTAGE",
    "INELIGIBLE_INSTRUCTOR",
    "INVALID_REFERENCE"
  ],
  "objective_breakdown": {
    "candidate_level_score": 877,
    "compactness": {
      "adjacent_pairs": 9,
      "points_per_pair": 15,
      "bonus

In [ ]:
OUTPUT_FILE = (
    ROOT / "tanseek_solver_output.json"
)

with open(
    OUTPUT_FILE,
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        solver_payload,
        f,
        indent=2,
        ensure_ascii=False
    )

print(
    "Saved:",
    OUTPUT_FILE.resolve()
)

Saved: C:\Users\engme\OneDrive\Desktop\Tanseek_Menna_Model_Starter\tanseek_solver_output.json


In [ ]:
# -------------------------
# Final Validation
# -------------------------

fixture_results = run_fixture_tests()

verify_published_immutable()

# All fixture tests must pass
assert fixture_results["pass"].all()

# Every required solver session must be either
# scheduled or explicitly unscheduled
assert (
    len(selected_schedule)
    + len(unscheduled_sessions)
    == len(weekly_sessions)
)

scheduled_keys = set(
    selected_schedule["session_key"]
)

unscheduled_keys = set(
    unscheduled_sessions
)

required_keys = set(
    weekly_sessions["session_key"]
)

# No session can be both scheduled and unscheduled
assert scheduled_keys.isdisjoint(
    unscheduled_keys
)

# No session can disappear
assert (
    scheduled_keys | unscheduled_keys
    == required_keys
)

# JSON file must exist
assert OUTPUT_FILE.exists()
assert len(post_solve_conflicts) == 0

print()
print("===================================")
print("TANSEEK MODEL FINAL CHECK ✅")
print("===================================")

print(
    "Fixture tests:",
    f"{fixture_results['pass'].sum()}/"
    f"{len(fixture_results)}"
)

print(
    "Existing allocations:",
    len(solver_existing_allocations)
)

print(
    "Weekly sessions to solve:",
    len(weekly_sessions)
)

print(
    "Scheduled:",
    len(selected_schedule)
)

print(
    "Unscheduled:",
    len(unscheduled_sessions)
)

print(
    "Term holidays loaded:",
    len(term_holidays)
)

print(
    "Room closures loaded:",
    len(room_closures)
)

print(
    "JSON output:",
    OUTPUT_FILE.resolve()
)
print(
    "Post-solve hard conflicts:",
    len(post_solve_conflicts)
)

print("===================================")

,case_id,expected,actual,all_reasons,pass
0,VALID_BASE,FEASIBLE,FEASIBLE,[],True
1,ROOM_COLLISION,ROOM_CONFLICT,ROOM_CONFLICT,[ROOM_CONFLICT],True
2,STAFF_COLLISION,STAFF_CONFLICT,STAFF_CONFLICT,[STAFF_CONFLICT],True
3,GROUP_COLLISION,GROUP_CONFLICT,GROUP_CONFLICT,[GROUP_CONFLICT],True
4,UNCONFIRMED,AVAILABILITY_NOT_CONFIRMED,AVAILABILITY_NOT_CONFIRMED,[AVAILABILITY_NOT_CONFIRMED],True
5,STAFF_UNAVAILABLE,STAFF_UNAVAILABLE,STAFF_UNAVAILABLE,[STAFF_UNAVAILABLE],True
6,EQUIPMENT_SHORTAGE,EQUIPMENT_SHORTAGE,EQUIPMENT_SHORTAGE,[EQUIPMENT_SHORTAGE],True
7,CAPACITY_SHORTAGE,CAPACITY_SHORTAGE,CAPACITY_SHORTAGE,[CAPACITY_SHORTAGE],True
8,WRONG_ROOM_TYPE,ROOM_TYPE_MISMATCH,ROOM_TYPE_MISMATCH,"[ROOM_TYPE_MISMATCH, EQUIPMENT_SHORTAGE]",True
9,BEYOND_HOURS,INVALID_DURATION,INVALID_DURATION,[INVALID_DURATION],True


All 11 fixture tests passed ✅
No published version yet — immutability check skipped 

TANSEEK MODEL FINAL CHECK ✅
Fixture tests: 11/11
Existing allocations: 1
Weekly sessions to solve: 24
Scheduled: 23
Unscheduled: 1
Term holidays loaded: 0
Room closures loaded: 0
JSON output: C:\Users\engme\OneDrive\Desktop\Tanseek_Menna_Model_Starter\tanseek_solver_output.json
Post-solve hard conflicts: 0


In [ ]:
# =================================================
# FINAL TANSEEK VALIDATION
# =================================================
# -------------------------
# Final Objective Metrics
# -------------------------

candidate_score_value = (
    int(selected_schedule["total_score"].sum())
    if not selected_schedule.empty
    else 0
)


selected_compact_pairs = int(
    sum(
        solver.Value(variable)
        for variable in compactness_vars
    )
)


compactness_bonus_value = (
    selected_compact_pairs
    * COMPACTNESS_POINTS
)


active_room_ids = [
    int(room_id)
    for room_id, variable in room_used.items()
    if solver.Value(variable) == 1
]


active_room_count = len(
    active_room_ids
)


room_utilization_penalty_value = (
    active_room_count
    * ROOM_ACTIVE_PENALTY
)


unscheduled_penalty_value = (
    len(unscheduled_sessions)
    * UNSCHEDULED_PENALTY
)


final_objective_value = (
    candidate_score_value
    + compactness_bonus_value
    - room_utilization_penalty_value
    - unscheduled_penalty_value
)


solver_objective_value = int(
    round(
        solver.ObjectiveValue()
    )
)


print(
    "Candidate score:",
    candidate_score_value
)

print(
    "Compact adjacent pairs:",
    selected_compact_pairs
)

print(
    "Compactness bonus:",
    compactness_bonus_value
)

print(
    "Active rooms:",
    active_room_count
)

print(
    "Room utilization penalty:",
    room_utilization_penalty_value
)

print(
    "Unscheduled penalty:",
    unscheduled_penalty_value
)

print(
    "Calculated objective:",
    final_objective_value
)

print(
    "Solver objective:",
    solver_objective_value
)


assert (
    final_objective_value
    == solver_objective_value
)

print(
    "Objective breakdown validated ✅"
)
print()
print("===================================")
print("TANSEEK FINAL VALIDATION")
print("===================================")


# -------------------------
# 1) Objective Breakdown
# -------------------------

assert (
    int(final_objective_value)
    == int(solver_objective_value)
), (
    f"Objective mismatch: "
    f"calculated={final_objective_value}, "
    f"solver={solver_objective_value}"
)

print("Objective breakdown validated ✅")


# -------------------------
# 2) Scheduled Payload
# -------------------------

assert isinstance(
    scheduled_payload,
    list
)

assert (
    len(scheduled_payload)
    == len(selected_schedule)
)

for item in scheduled_payload:

    assert "session_key" in item
    assert "score" in item
    assert "score_breakdown" in item
    assert "alternatives" in item

    breakdown = item[
        "score_breakdown"
    ]

    assert "staff_preference" in breakdown
    assert "capacity_fit" in breakdown
    assert "equipment_match" in breakdown
    assert "candidate_total" in breakdown


print("Scheduled payload ready ✅")


# -------------------------
# 3) Hard Conflict Check
# -------------------------

assert (
    len(post_solve_conflicts)
    == 0
), post_solve_conflicts

print("Post-solve hard conflicts: 0 ✅")


# -------------------------
# 4) Fixture Tests
# -------------------------

fixture_results = run_fixture_tests()

assert (
    fixture_results["pass"].all()
)

print(
    f"Fixture tests: "
    f"{fixture_results['pass'].sum()}/"
    f"{len(fixture_results)} ✅"
)


# -------------------------
# 5) Session Coverage
# -------------------------

scheduled_keys = set(
    selected_schedule[
        "session_key"
    ]
)

unscheduled_keys = set(
    unscheduled_sessions
)

required_keys = set(
    weekly_sessions[
        "session_key"
    ]
)


assert scheduled_keys.isdisjoint(
    unscheduled_keys
)

assert (
    scheduled_keys
    | unscheduled_keys
    == required_keys
)


print("Session coverage validated ✅")


# -------------------------
# 6) Output JSON
# -------------------------

assert OUTPUT_FILE.exists()

print("JSON output exists ✅")


# -------------------------
# FINAL RESULT
# -------------------------

print()
print("===================================")
print("TANSEEK MODEL FINAL CHECK ✅")
print("===================================")

print(
    "Existing allocations:",
    len(solver_existing_allocations)
)

print(
    "Weekly sessions to solve:",
    len(weekly_sessions)
)

print(
    "Scheduled:",
    len(selected_schedule)
)

print(
    "Unscheduled:",
    len(unscheduled_sessions)
)

print(
    "Candidate-level score:",
    candidate_score_value
)

print(
    "Compact adjacent pairs:",
    selected_compact_pairs
)

print(
    "Compactness bonus:",
    compactness_bonus_value
)

print(
    "Active rooms:",
    active_room_count
)

print(
    "Room utilization penalty:",
    room_utilization_penalty_value
)

print(
    "Final objective:",
    final_objective_value
)

print(
    "Term holidays loaded:",
    len(term_holidays)
)

print(
    "Room closures loaded:",
    len(room_closures)
)

print(
    "JSON output:",
    OUTPUT_FILE.resolve()
)

print("===================================")

Candidate score: 877
Compact adjacent pairs: 9
Compactness bonus: 135
Active rooms: 4
Room utilization penalty: 40
Unscheduled penalty: 100000
Calculated objective: -99028
Solver objective: -99028
Objective breakdown validated ✅

TANSEEK FINAL VALIDATION
Objective breakdown validated ✅
Scheduled payload ready ✅
Post-solve hard conflicts: 0 ✅


,case_id,expected,actual,all_reasons,pass
0,VALID_BASE,FEASIBLE,FEASIBLE,[],True
1,ROOM_COLLISION,ROOM_CONFLICT,ROOM_CONFLICT,[ROOM_CONFLICT],True
2,STAFF_COLLISION,STAFF_CONFLICT,STAFF_CONFLICT,[STAFF_CONFLICT],True
3,GROUP_COLLISION,GROUP_CONFLICT,GROUP_CONFLICT,[GROUP_CONFLICT],True
4,UNCONFIRMED,AVAILABILITY_NOT_CONFIRMED,AVAILABILITY_NOT_CONFIRMED,[AVAILABILITY_NOT_CONFIRMED],True
5,STAFF_UNAVAILABLE,STAFF_UNAVAILABLE,STAFF_UNAVAILABLE,[STAFF_UNAVAILABLE],True
6,EQUIPMENT_SHORTAGE,EQUIPMENT_SHORTAGE,EQUIPMENT_SHORTAGE,[EQUIPMENT_SHORTAGE],True
7,CAPACITY_SHORTAGE,CAPACITY_SHORTAGE,CAPACITY_SHORTAGE,[CAPACITY_SHORTAGE],True
8,WRONG_ROOM_TYPE,ROOM_TYPE_MISMATCH,ROOM_TYPE_MISMATCH,"[ROOM_TYPE_MISMATCH, EQUIPMENT_SHORTAGE]",True
9,BEYOND_HOURS,INVALID_DURATION,INVALID_DURATION,[INVALID_DURATION],True


All 11 fixture tests passed ✅
Fixture tests: 11/11 ✅
Session coverage validated ✅
JSON output exists ✅

TANSEEK MODEL FINAL CHECK ✅
Existing allocations: 1
Weekly sessions to solve: 24
Scheduled: 23
Unscheduled: 1
Candidate-level score: 877
Compact adjacent pairs: 9
Compactness bonus: 135
Active rooms: 4
Room utilization penalty: 40
Final objective: -99028
Term holidays loaded: 0
Room closures loaded: 0
JSON output: C:\Users\engme\OneDrive\Desktop\Tanseek_Menna_Model_Starter\tanseek_solver_output.json
